# Target Karşılaştırma — A vs B vs C vs Mevcut

**Tarih:** 2026-05-12
**Amaç:** V6 modelinde %86 pozitif dengesizliği çözmek için 3 alternatif target tanımını ampirik olarak test et.

## Test edilen target tanımları

| Kod | Tanım | SONUCTIPI dahil | Beklenen pozitif oran |
|---|---|---|---|
| **MEVCUT** | Geniş (4 kategori) | Çekilerek + Oto Değ + Yol Ust-Garaj + Telefonla-Garaj | ~%86 |
| **(A)** | V5 dar (2 kategori) | Çekilerek + Oto Değişimi | ~%60 |
| **(B)** | 3 kategori | Çekilerek + Oto Değ + Yol Ust-Garaj | ~%70 |
| **(C)** | Sayı eşiği | Q2'de ≥3 ciddi (mevcut) arıza | ~%50 |

## Değerlendirme kriterleri
1. **Pozitif oran** (dengeli olması iyi → %30-%70 ideal)
2. **Korelasyon gücü** (feature'lar target ile daha güçlü r → daha iyi)
3. **Zayıf sinyaller restore** mu? (yakit_turu_cng, dur_kalk_index)
4. **Pos vs Neg ayrımı** (ortalamalar arasında belirgin fark → ayırt edici target)

---
## 1. Veri Yükleme — Q2 SONUCTIPI'lı ham arıza

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Q1 features (mevcut hazır CSV)
feat = pd.read_csv('features_final_v6_q1.csv', encoding='utf-8-sig')
print(f'Q1 features: {feat.shape}')
print(f'Mevcut target_q2 pozitif oran: {feat["target_q2"].mean()*100:.1f}%')

# Ham arıza (SONUCTIPI dahil)
at = pd.read_csv('../panel_data/temiz_veri/ariza_temiz.csv', low_memory=False,
                 usecols=['KAPINO','SONUCTIPI','OLAYTARIHI','YAKITTURU','MODEL','ciddi_ariza'])
at['OLAYTARIHI'] = pd.to_datetime(at['OLAYTARIHI'], format='mixed')

# Aynı YAKITTURU filtresi (V6 prep ile tutarlılık)
mask_b = at['YAKITTURU']=='Bilinmiyor'
at.loc[mask_b & at['MODEL'].astype(str).str.contains('CNG', na=False), 'YAKITTURU'] = 'CNG'
at.loc[mask_b & ~at['MODEL'].astype(str).str.contains('CNG', na=False), 'YAKITTURU'] = 'MOTORIN'
at = at[at['YAKITTURU'].isin(['MOTORIN','CNG'])].copy()

# Q2 filtresi
SPLIT = pd.Timestamp('2025-04-01')
END_Q2 = pd.Timestamp('2025-07-01')
q2 = at[(at['OLAYTARIHI']>=SPLIT) & (at['OLAYTARIHI']<END_Q2)].copy()
print(f'\nQ2 arıza: {len(q2):,}, araç: {q2["KAPINO"].nunique():,}')
print(f'\nQ2 SONUCTIPI dağılım:')
print(q2['SONUCTIPI'].value_counts())

Q1 features: (3508, 27)
Mevcut target_q2 pozitif oran: 86.3%

Q2 arıza: 31,126, araç: 3,483

Q2 SONUCTIPI dağılım:
SONUCTIPI
Yol Ustası - Servis    10936
Telefonla - Servis      6744
Yol Ustası - Garaj      5850
Oto Değişimi            2620
Çekilerek - Garaj       2053
Telefonla - Garaj       1670
Kayıtçı - Garaj         1253
Name: count, dtype: int64


---
## 2. 4 Target Tanımı Hesaplama

In [2]:
# SONUCTIPI etiketleri (Q2 datasında görünen formuyla)
TIP_CEK = 'Çekilerek - Garaj'
TIP_OTO = 'Oto Değişimi'
TIP_YOL_G = 'Yol Ustası - Garaj'
TIP_TEL_G = 'Telefonla - Garaj'

# Etiket flag'leri
q2['is_A'] = q2['SONUCTIPI'].isin([TIP_CEK, TIP_OTO]).astype(int)
q2['is_B'] = q2['SONUCTIPI'].isin([TIP_CEK, TIP_OTO, TIP_YOL_G]).astype(int)
q2['is_MEVCUT'] = q2['SONUCTIPI'].isin([TIP_CEK, TIP_OTO, TIP_YOL_G, TIP_TEL_G]).astype(int)

# Araç bazında agg
tg = q2.groupby('KAPINO').agg(
    target_A=('is_A','max'),
    n_A=('is_A','sum'),
    target_B=('is_B','max'),
    n_B=('is_B','sum'),
    target_MEVCUT=('is_MEVCUT','max'),
    n_MEVCUT=('is_MEVCUT','sum'),
).reset_index()

# Features'a merge (Q2'de hiç görünmeyen araçlar = 0)
feat2 = feat.merge(tg, on='KAPINO', how='left')
for c in ['target_A','target_B','target_MEVCUT','n_A','n_B','n_MEVCUT']:
    feat2[c] = feat2[c].fillna(0).astype(int)

# (C) Sayı eşiği — q2_ciddi_n (mevcut wide ciddi sayısı) >= 3
feat2['target_C'] = (feat2['q2_ciddi_n'] >= 3).astype(int)

# Pozitif oran tablosu
print('=== POZİTİF ORAN ===')
print(f'{"Target":<15s} {"Pozitif":>10s} {"Toplam":>10s} {"Oran":>8s}')
print('-'*50)
for name, col in [('MEVCUT (4 kat)','target_MEVCUT'),('(A) V5 dar','target_A'),('(B) 3 kategori','target_B'),('(C) Sayı ≥3','target_C')]:
    pos = feat2[col].sum()
    tot = len(feat2)
    print(f'{name:<15s} {pos:>10,} {tot:>10,} {pos/tot*100:>7.1f}%')

=== POZİTİF ORAN ===
Target             Pozitif     Toplam     Oran
--------------------------------------------------
MEVCUT (4 kat)       3,028      3,508    86.3%
(A) V5 dar           2,339      3,508    66.7%
(B) 3 kategori       2,949      3,508    84.1%
(C) Sayı ≥3          1,893      3,508    54.0%


---
## 3. Korelasyon Karşılaştırma — Feature × Her Target

In [3]:
FEATURES = ['yas','egim_maruziyet','garaj_sistem_lift','garaj_marka_lift',
            'yakit_turu_cng','verimsizlik_skoru','hat_zorluk','cascade_risk_skor',
            'farkli_sofor_sayisi','dur_kalk_index','ariza_q1','gecmis_ciddi_oran']

TARGETS = [('MEVCUT','target_MEVCUT'),('A','target_A'),('B','target_B'),('C','target_C')]

rows = []
for f in FEATURES:
    row = {'feature': f}
    for tn, tc in TARGETS:
        r, p = stats.pearsonr(feat2[f], feat2[tc])
        row[f'r_{tn}'] = r
        row[f'p_{tn}'] = p
    rows.append(row)

corr_df = pd.DataFrame(rows)

# Sade görüntü (sadece r değerleri + significance flag)
print('=== FEATURE × TARGET KORELASYON (Pearson r) ===')
print(f'{"Feature":<25s} {"MEVCUT":>10s} {"(A)":>10s} {"(B)":>10s} {"(C)":>10s}')
print('-'*70)
for _, r in corr_df.iterrows():
    # Mark significant (p<0.05) with *, strong (|r|>0.2) with **
    def mark(rv, pv):
        s = f'{rv:+.3f}'
        if abs(rv) > 0.2: s += '**'
        elif pv < 0.05: s += '*'
        return s
    print(f'{r["feature"]:<25s} {mark(r["r_MEVCUT"],r["p_MEVCUT"]):>11s} {mark(r["r_A"],r["p_A"]):>11s} {mark(r["r_B"],r["p_B"]):>11s} {mark(r["r_C"],r["p_C"]):>11s}')
print('\n** : |r|>0.2 (güçlü) | * : p<0.05 (anlamlı)')

=== FEATURE × TARGET KORELASYON (Pearson r) ===
Feature                       MEVCUT        (A)        (B)        (C)
----------------------------------------------------------------------
yas                          +0.222**     +0.150*    +0.222**     +0.122*
egim_maruziyet                +0.162*     +0.078*     +0.179*     +0.158*
garaj_sistem_lift             +0.147*     +0.073*     +0.164*     +0.168*
garaj_marka_lift             +0.317**     +0.115*    +0.315**    +0.323**
yakit_turu_cng                 -0.022     +0.048*     -0.040*     -0.118*
verimsizlik_skoru            +0.208**     +0.035*    +0.215**    +0.325**
hat_zorluk                   +0.238**      -0.006    +0.264**    +0.347**
cascade_risk_skor             +0.133*     +0.037*     +0.125*     +0.158*
farkli_sofor_sayisi          +0.254**     +0.045*    +0.250**    +0.372**
dur_kalk_index                 -0.029     -0.073*     -0.061*     -0.058*
ariza_q1                     +0.268**     +0.069*    +0.259**    +0.353

---
## 4. Anlamlı Feature Sayısı + Ortalama |r|

In [4]:
print('=== TARGET KALİTE METRİKLERİ ===')
print(f'{"Target":<15s} {"Anlamlı(p<.05)":>15s} {"Güçlü(|r|>0.2)":>15s} {"Ortalama |r|":>14s} {"Max |r|":>10s}')
print('-'*80)
for tn, tc in TARGETS:
    sig = (corr_df[f'p_{tn}'] < 0.05).sum()
    strong = (corr_df[f'r_{tn}'].abs() > 0.2).sum()
    mean_abs = corr_df[f'r_{tn}'].abs().mean()
    max_abs = corr_df[f'r_{tn}'].abs().max()
    print(f'{tn:<15s} {sig:>10d}/{len(FEATURES)} {strong:>10d}/{len(FEATURES)} {mean_abs:>14.3f} {max_abs:>10.3f}')

print('\n— Yorumlama —')
print('• Anlamlı feature sayısı yüksek → target öğrenilebilir sinyaller içeriyor')
print('• Ortalama |r| yüksek → genel olarak feature\'lar daha ayırt edici')
print('• Pozitif oran ~%50 + güçlü korelasyon → ideal kombinasyon')

=== TARGET KALİTE METRİKLERİ ===
Target           Anlamlı(p<.05)  Güçlü(|r|>0.2)   Ortalama |r|    Max |r|
--------------------------------------------------------------------------------
MEVCUT                  10/12          6/12          0.178      0.317
A                       11/12          0/12          0.066      0.150
B                       12/12          6/12          0.186      0.315
C                       12/12          5/12          0.221      0.372

— Yorumlama —
• Anlamlı feature sayısı yüksek → target öğrenilebilir sinyaller içeriyor
• Ortalama |r| yüksek → genel olarak feature'lar daha ayırt edici
• Pozitif oran ~%50 + güçlü korelasyon → ideal kombinasyon


---
## 5. Zayıf Sinyaller Restore Testi — yakit_turu_cng, dur_kalk_index

In [5]:
WEAK = ['yakit_turu_cng','dur_kalk_index']
print('=== ZAYIF SİNYAL RESTORE TESTİ ===\n')
for f in WEAK:
    print(f'--- {f} ---')
    print(f'{"Target":<10s} {"r":>10s} {"p":>10s} {"Pos mean":>12s} {"Neg mean":>12s} {"Fark":>10s}')
    print('-'*70)
    for tn, tc in TARGETS:
        r, p = stats.pearsonr(feat2[f], feat2[tc])
        pos_m = feat2.loc[feat2[tc]==1, f].mean()
        neg_m = feat2.loc[feat2[tc]==0, f].mean()
        diff = pos_m - neg_m
        flag = ' ✓' if p < 0.05 else ''
        print(f'{tn:<10s} {r:>+10.4f} {p:>10.4f} {pos_m:>12.4f} {neg_m:>12.4f} {diff:>+10.4f}{flag}')
    print()

=== ZAYIF SİNYAL RESTORE TESTİ ===

--- yakit_turu_cng ---
Target              r          p     Pos mean     Neg mean       Fark
----------------------------------------------------------------------
MEVCUT        -0.0224     0.1838       0.0971       0.1167    -0.0196
A             +0.0477     0.0047       0.1099       0.0796    +0.0303 ✓
B             -0.0396     0.0191       0.0946       0.1270    -0.0324 ✓
C             -0.1181     0.0000       0.0671       0.1381    -0.0710 ✓

--- dur_kalk_index ---
Target              r          p     Pos mean     Neg mean       Fark
----------------------------------------------------------------------
MEVCUT        -0.0290     0.0861    3659.3242    3843.5722  -184.2480
A             -0.0728     0.0000    3572.0695    3909.5619  -337.4925 ✓
B             -0.0612     0.0003    3626.3178    3991.6588  -365.3410 ✓
C             -0.0577     0.0006    3568.1602    3820.9418  -252.7816 ✓



---
## 6. Pos vs Neg Ortalamaları — Tüm Feature'lar

In [6]:
# Her target için pos/neg ortalamaları + Cohen's d benzeri standardize fark
print('=== POS vs NEG STANDARDİZE FARK (|μ_pos - μ_neg| / σ) ===')
print(f'{"Feature":<25s} {"MEVCUT":>10s} {"(A)":>10s} {"(B)":>10s} {"(C)":>10s}')
print('-'*70)
d_rows = []
for f in FEATURES:
    std = feat2[f].std()
    if std < 1e-9:
        continue
    row = {'feature': f}
    vals = []
    for tn, tc in TARGETS:
        pos_m = feat2.loc[feat2[tc]==1, f].mean()
        neg_m = feat2.loc[feat2[tc]==0, f].mean()
        d = (pos_m - neg_m) / std
        row[f'd_{tn}'] = d
        vals.append(f'{d:>+10.3f}')
    d_rows.append(row)
    print(f'{f:<25s} {vals[0]:>11s} {vals[1]:>11s} {vals[2]:>11s} {vals[3]:>11s}')

d_df = pd.DataFrame(d_rows)
print(f'\nOrtalama |d|:')
for tn, _ in TARGETS:
    print(f'  {tn}: {d_df[f"d_{tn}"].abs().mean():.3f}')

=== POS vs NEG STANDARDİZE FARK (|μ_pos - μ_neg| / σ) ===
Feature                       MEVCUT        (A)        (B)        (C)
----------------------------------------------------------------------
yas                            +0.646      +0.318      +0.607      +0.244
egim_maruziyet                 +0.471      +0.166      +0.488      +0.317
garaj_sistem_lift              +0.426      +0.156      +0.447      +0.337
garaj_marka_lift               +0.922      +0.244      +0.862      +0.648
yakit_turu_cng                 -0.065      +0.101      -0.108      -0.237
verimsizlik_skoru              +0.607      +0.075      +0.587      +0.653
hat_zorluk                     +0.693      -0.013      +0.720      +0.696
cascade_risk_skor              +0.387      +0.079      +0.342      +0.317
farkli_sofor_sayisi            +0.739      +0.096      +0.683      +0.745
dur_kalk_index                 -0.084      -0.154      -0.167      -0.116
ariza_q1                       +0.779      +0.146      +0.709

---
## 7. ÖZET TABLO ve TAVSİYE

In [7]:
# Karar matrisi
print('=== KARAR MATRİSİ ===\n')
print(f'{"Kriter":<35s} {"MEVCUT":>10s} {"(A)":>10s} {"(B)":>10s} {"(C)":>10s}')
print('='*80)
rows_sum = []
for tn, tc in TARGETS:
    pos_rate = feat2[tc].mean()
    sig = (corr_df[f'p_{tn}'] < 0.05).sum()
    strong = (corr_df[f'r_{tn}'].abs() > 0.2).sum()
    mean_abs = corr_df[f'r_{tn}'].abs().mean()
    cng_p = corr_df.loc[corr_df['feature']=='yakit_turu_cng', f'p_{tn}'].values[0]
    dur_p = corr_df.loc[corr_df['feature']=='dur_kalk_index', f'p_{tn}'].values[0]
    mean_d = d_df[f'd_{tn}'].abs().mean()
    rows_sum.append({
        'target': tn, 'pos_rate': pos_rate, 'sig': sig, 'strong': strong,
        'mean_abs_r': mean_abs, 'cng_sig': cng_p<0.05, 'dur_sig': dur_p<0.05,
        'mean_abs_d': mean_d,
    })

s = pd.DataFrame(rows_sum)

def fmt(idx, key, fmt_str):
    vals = [fmt_str.format(s.iloc[i][key]) for i in range(4)]
    return ' '.join(f'{v:>11s}' for v in vals)

print(f'{"Pozitif oran (% — ideal ~50)":<35s} {fmt(0,"pos_rate","{:>.1%}")}')
print(f'{"Anlamlı feature sayısı (/12)":<35s} {fmt(0,"sig","{:>d}")}')
print(f'{"Güçlü |r|>0.2 (/12)":<35s} {fmt(0,"strong","{:>d}")}')
print(f'{"Ortalama |r|":<35s} {fmt(0,"mean_abs_r","{:>.3f}")}')
print(f'{"Ortalama |Cohen d|":<35s} {fmt(0,"mean_abs_d","{:>.3f}")}')
print(f'{"yakit_turu_cng restore":<35s} {fmt(0,"cng_sig","{!s}")}')
print(f'{"dur_kalk_index restore":<35s} {fmt(0,"dur_sig","{!s}")}')

# Skor: pozitif oran 0.5'e yakınlık (1 - |0.5 - p|*2) + ortalama |r| * 3 + restore bonusları
s['balance_score'] = 1 - (s['pos_rate'] - 0.5).abs() * 2
s['signal_score'] = s['mean_abs_r'] * 3
s['restore_bonus'] = s['cng_sig'].astype(int)*0.1 + s['dur_sig'].astype(int)*0.1
s['total'] = s['balance_score'] + s['signal_score'] + s['restore_bonus']

print('\n=== TOPLAM SKOR ===')
for _, r in s.sort_values('total', ascending=False).iterrows():
    print(f'  {r["target"]:<10s}: balance={r["balance_score"]:.2f} + signal={r["signal_score"]:.2f} + restore={r["restore_bonus"]:.2f} = TOPLAM {r["total"]:.2f}')

winner = s.sort_values('total', ascending=False).iloc[0]['target']
print(f'\n>>> KAZANAN: {winner} <<<')
print('\n(Kullanıcı son kararı kendisi verir, bu skor heuristic — kapsamlı bakış için tüm tablolara bak.)')

=== KARAR MATRİSİ ===

Kriter                                  MEVCUT        (A)        (B)        (C)
Pozitif oran (% — ideal ~50)              86.3%       66.7%       84.1%       54.0%
Anlamlı feature sayısı (/12)                 10          11          12          12
Güçlü |r|>0.2 (/12)                           6           0           6           5
Ortalama |r|                              0.178       0.066       0.186       0.221
Ortalama |Cohen d|                        0.517       0.141       0.508       0.443
yakit_turu_cng restore                    False        True        True        True
dur_kalk_index restore                    False        True        True        True

=== TOPLAM SKOR ===
  C         : balance=0.92 + signal=0.66 + restore=0.20 = TOPLAM 1.78
  B         : balance=0.32 + signal=0.56 + restore=0.20 = TOPLAM 1.08
  A         : balance=0.67 + signal=0.20 + restore=0.20 = TOPLAM 1.07
  MEVCUT    : balance=0.27 + signal=0.53 + restore=0.00 = TOPLAM 0.81

>>> KAZ

---
## 8. Detay Karşılaştırma — Pozitif Sınıf Dağılımı

Her target için pozitif olan araçların özelliklerine bakalım — gerçekten 'riskli' araçları yakalıyor mu?

In [8]:
print('=== POZİTİF SINIF PROFİLİ ===')
print(f'{"Target":<10s} {"Pos n":>8s} {"Yas ort":>10s} {"Q1 ariza":>10s} {"Q1 ciddi%":>12s} {"CNG%":>8s}')
print('-'*70)
for tn, tc in TARGETS:
    pos = feat2[feat2[tc]==1]
    print(f'{tn:<10s} {len(pos):>8,} {pos["yas"].mean():>10.1f} {pos["ariza_q1"].mean():>10.1f} {pos["gecmis_ciddi_oran"].mean()*100:>11.1f}% {pos["yakit_turu_cng"].mean()*100:>7.1f}%')

print('\n=== NEGATİF SINIF (Q2 hiç ariza yapmayan) ===')
for tn, tc in TARGETS:
    neg_noariza = feat2[(feat2[tc]==0) & (feat2['q2_ariza_n']==0)]
    neg_ariza = feat2[(feat2[tc]==0) & (feat2['q2_ariza_n']>0)]
    print(f'  {tn}: {len(neg_noariza)} hiç arıza yok | {len(neg_ariza)} arıza yapmış ama target=0')

=== POZİTİF SINIF PROFİLİ ===
Target        Pos n    Yas ort   Q1 ariza    Q1 ciddi%     CNG%
----------------------------------------------------------------------
MEVCUT        3,028       12.0        8.4        36.2%     9.7%
A             2,339       12.1        8.1        36.1%    11.0%
B             2,949       12.0        8.4        36.4%     9.5%
C             1,893       12.1        9.5        38.2%     6.7%

=== NEGATİF SINIF (Q2 hiç ariza yapmayan) ===
  MEVCUT: 56 hiç arıza yok | 424 arıza yapmış ama target=0
  A: 56 hiç arıza yok | 1113 arıza yapmış ama target=0
  B: 56 hiç arıza yok | 503 arıza yapmış ama target=0
  C: 56 hiç arıza yok | 1559 arıza yapmış ama target=0


---
## 9. CSV Export — Her Target Variantı

ML model aşamasında kolayca yüklenebilmesi için her 4 target tek CSV'de tutuluyor.

In [9]:
out_cols = [c for c in feat2.columns if c not in ['target_q2']]  # mevcut target_q2 zaten target_MEVCUT'a denk
out = feat2[out_cols].copy()
out.to_csv('features_final_v6_q1_4target.csv', index=False, encoding='utf-8-sig')
print(f'Export: features_final_v6_q1_4target.csv ({len(out):,} araç × {len(out.columns)} kolon)')
print(f'Target kolonları: target_MEVCUT, target_A, target_B, target_C')
print(f'Sayı kolonları: n_MEVCUT, n_A, n_B, q2_ciddi_n (C için)')

Export: features_final_v6_q1_4target.csv (3,508 araç × 33 kolon)
Target kolonları: target_MEVCUT, target_A, target_B, target_C
Sayı kolonları: n_MEVCUT, n_A, n_B, q2_ciddi_n (C için)
